# Question 4. Adaptive Facility Location

In [1]:
import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB

# quiet Gurobi environment shared by all models
env = gp.Env(params={"OutputFlag": 0})

# load data
fac = pd.read_csv("HW4_data/facilities.csv")
c = fac["cost"].values                                                   # construction cost
cap = fac["capacity"].values                                             # capacity
q = pd.read_csv("HW4_data/production.csv", index_col=0).values           # production cost (20 x 500)
t = pd.read_csv("HW4_data/transportation.csv", index_col=0).values       # transport cost (50 x 20)
u = pd.read_csv("HW4_data/zones.csv")["unmet"].values                    # unmet demand penalty
D = pd.read_csv("HW4_data/demand.csv", index_col=0).values               # demand (50 x 500)

n, m, S = len(c), len(u), D.shape[1]
p = 1 / S
print(f"{n} facilities, {m} zones, {S} scenarios")

20 facilities, 50 zones, 500 scenarios


## Part (a): Formulation

Let $J=\{1,\dots,20\}$ be the candidate facilities, $I=\{1,\dots,50\}$ the customer zones, and $s=1,\dots,S$ the $S=500$ scenarios, each with probability $p_s = 1/S$.

**First-stage variable** which is chosen before the scenario is known, shared by all scenarios:
- $x_j \in \{0,1\}$: 1 if facility $j$ is built.

**Second-stage variables** which is chosen after scenario $s$ is observed:
- $y_{js} \ge 0$: amount produced at facility $j$ in scenario $s$
- $z_{ijs} \ge 0$: amount shipped from facility $j$ to zone $i$ in scenario $s$
- $w_{is} \ge 0$: unmet demand in zone $i$ in scenario $s$


$$
\begin{aligned}
\min_{x,y,z,w}\quad & \sum_{j \in J} c_j x_j \;+\; \sum_{s=1}^{S} p_s \Big( \sum_{j \in J} q_{js}\, y_{js} + \sum_{i \in I}\sum_{j \in J} t_{ij}\, z_{ijs} + \sum_{i \in I} u_i\, w_{is} \Big) \\
\text{s.t.}\quad & y_{js} \le C_j\, x_j && \forall j \in J,\ s \\
& \sum_{i \in I} z_{ijs} = y_{js} && \forall j \in J,\ s \\
& \sum_{j \in J} z_{ijs} + w_{is} = D_{is} && \forall i \in I,\ s \\
& x_j \in \{0,1\},\quad y_{js},\, z_{ijs},\, w_{is} \ge 0 && \forall i, j, s
\end{aligned}
$$


A couple notes. 
- We can see that construction cost is paid once; production, transportation and unmet-demand costs are averaged over the scenarios.
- For the capacity constraint, a facility can produce up to $C_j$ only if it is built. A closed facility ($x_j=0$) produces nothing.
- Everything produced at $j$ is shipped out. Since production costs are nonnegative, producing more than is shipped never helps, so using $\sum_i z_{ijs} \le y_{js}$ would give the same optimal cost.

## Part (b): Adaptive stochastic model

In [2]:
model = gp.Model(env=env)

build = model.addVars(n, vtype=GRB.BINARY)
produce = model.addVars(n, S, lb=0)
ship = model.addVars(m, n, S, lb=0)
unmet = model.addVars(m, S, lb=0)

# construction cost once + expected second-stage cost
model.setObjective(
    gp.quicksum(c[j] * build[j] for j in range(n))
    + p * gp.quicksum(q[j, s] * produce[j, s] for j in range(n) for s in range(S))
    + p * gp.quicksum(t[i, j] * ship[i, j, s] for i in range(m) for j in range(n) for s in range(S))
    + p * gp.quicksum(u[i] * unmet[i, s] for i in range(m) for s in range(S)),
    GRB.MINIMIZE)

for s in range(S):
    for j in range(n):
        # capacity, only if built
        model.addConstr(produce[j, s] <= cap[j] * build[j])
        # ship out what is produced
        model.addConstr(gp.quicksum(ship[i, j, s] for i in range(m)) == produce[j, s])
    for i in range(m):
        # demand is served or unmet
        model.addConstr(gp.quicksum(ship[i, j, s] for j in range(n)) + unmet[i, s] == D[i, s])

model.Params.MIPGap = 1e-6
model.optimize()
print("Status optimal:", model.Status == GRB.OPTIMAL)

RP = model.ObjVal
built_rp = [j + 1 for j in range(n) if build[j].X > 0.5]
print(f"Optimal expected total cost (RP): {RP:.4f}")
print("Facilities built:", built_rp)

Status optimal: True
Optimal expected total cost (RP): 1206.7856
Facilities built: [1, 2, 8, 10, 11, 12, 13, 15, 16, 18, 19, 20]


## Part (c):

In [ ]:
def solve_single(q_s, D_s):
    '''
    Solve the single-scenario problem with production costs q_s and demands D_s.
    Returns the status, objective value, and construction decisions.
    '''
    mod = gp.Model(env=env)
    b = mod.addVars(n, vtype=GRB.BINARY)
    y = mod.addVars(n, lb=0)
    z = mod.addVars(m, n, lb=0)
    w = mod.addVars(m, lb=0)
    mod.setObjective(
        gp.quicksum(c[j] * b[j] + q_s[j] * y[j] for j in range(n))
        + gp.quicksum(t[i, j] * z[i, j] for i in range(m) for j in range(n))
        + gp.quicksum(u[i] * w[i] for i in range(m)),
        GRB.MINIMIZE)
    for j in range(n):
        mod.addConstr(y[j] <= cap[j] * b[j])
        mod.addConstr(gp.quicksum(z[i, j] for i in range(m)) == y[j])
    for i in range(m):
        mod.addConstr(gp.quicksum(z[i, j] for j in range(n)) + w[i] == D_s[i])
    mod.Params.MIPGap = 1e-6
    mod.optimize()
    return mod.Status, mod.ObjVal, [b[j].X for j in range(n)]

# solve the expected-value problem
status, EV, build_ev = solve_single(q.mean(axis=1), D.mean(axis=1))
print("EV status optimal:", status == GRB.OPTIMAL)
print(f"EV objective (average scenario): {EV:.4f}")
print("Facilities built (EV):", [j + 1 for j in range(n) if build_ev[j] > 0.5])

# fix the EV construction decisions in the stochastic model and let the second stage adapt
for j in range(n):
    build[j].LB = round(build_ev[j])
    build[j].UB = round(build_ev[j])
model.optimize()
print("EEV status optimal:", model.Status == GRB.OPTIMAL)

EEV = model.ObjVal
print(f"Expected cost of EV decisions (EEV): {EEV:.4f}")

EV status optimal: True
EV objective (average scenario): 698.7323
Facilities built (EV): [1, 2, 10, 11, 16, 18, 19]


EEV status optimal: True
Expected cost of EV decisions (EEV): 1427.0976


### Wait-and-see benchmark

Solve each scenario on its own (construction decisions can differ by scenario), then average the optimal costs.

In [4]:
ws_costs = []
n_optimal = 0
for s in range(S):
    status, obj, _ = solve_single(q[:, s], D[:, s])
    n_optimal += (status == GRB.OPTIMAL)
    ws_costs.append(obj)

WS = np.mean(ws_costs)
print(f"Scenarios solved to optimality: {n_optimal}/{S}")
print(f"Wait-and-see cost (WS): {WS:.4f}")

Scenarios solved to optimality: 500/500
Wait-and-see cost (WS): 754.5528


### EVPI and VSS

In [5]:
EVPI = RP - WS
VSS = EEV - RP

print("WS <= RP <= EEV:", WS <= RP + 1e-6 and RP <= EEV + 1e-6)

results = pd.DataFrame({
    "Value": [RP, EEV, WS, EVPI, VSS, EVPI / RP * 100, VSS / EEV * 100]
}, index=["RP", "EEV", "WS", "EVPI", "VSS", "Relative EVPI (%)", "Relative VSS (%)"])
results.round(4)

WS <= RP <= EEV: True


,Value
RP,1206.7856
EEV,1427.0976
WS,754.5528
EVPI,452.2328
VSS,220.3120
Relative EVPI (%),37.4742
Relative VSS (%),15.4378


**Interpretation.** The relative EVPI is approximately 37.5%, indicating that perfect information about future demand and production costs could reduce expected costs from 1206.79 to 754.55. The relative VSS is approximately 15.4%. This means that accounting for uncertainty when making construction decisions saves 220.31 in expected costs compared with using the expected-value solution. The EV model builds only 7 facilities based on average demand and production costs, whereas the stochastic model builds 12. 